In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
from scipy.stats import norm
from scipy.stats import binomtest

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

#For ML, seems like SpecSup250, Per90, and SpecSup250 + Per90 are the way to go
#If we decide to do the whole controlling for effect size when assigning to genes/species for NC PhyloP, we will need to alter code to do it for just species for ML

gobp = pd.read_csv("GOBP_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("HPO_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
d_abrev = {"LiangSteinNeuron":"FC exc. neur.", "FetalChondrocytes":"F chond.", "SertoliMale":"FG sertoli", "preGC_IIaFemale":"FG preGC IIa",\
      "NeuralFemale":"FG neur.", "FetalGonadImmuneFemale":"FG immune", "VIP":"AC VIP inh. neur.", "LiangSteinProgenitor":"FC prog.",\
      "AdultHeartVentricularCardiomyocyte":"AH cardiomyo.", "AdultLoopOfHenle":"AK loop of henle", "FetalBrainNeurGlioblast_CB_VZ":"FCB glioblast",\
     "AdultProximalTubule":"AK prox. tub.", "FetalLeydigMale":"FG leydig", "SST":"AC SST inh neur.", "KosoyRoussosControlMicroglia":"AC microglia",\
     "FetalBrainFloorPlate":"FB fl. plate", "FetalArterialECs":"FH endoth.", "ASCT":"AC astro.", "FetalBrainCOP":"FB COP",\
     "AMY":"AA neur.", "PVALB":"AC PVALB inh neur.", "ITL23":"AC L2-3 IT neur.", "FetalBrainNeurCB_GNP_IPC_1":"FB inter. prog.", "FetalBrainNeurDAergic":"FB DA neur.",\
      "OGC":"AC Oligo.", "D1Pu":"AP D1 inh neur.", "FetalBrainNeurSerotonergic":"FB 5-HT neur.", "FetalBrainNeurDRG_2":"FS DRG neur.",\
      "FetalHeartPericytes":"FH peri.", "FetalHeartEndocardium":"FH endocard.", "FetalHeartCardiacFibroblasts":"FH fibro.", "FetalBrainNeurPurkinje_6":"FCB Purk. inh neur.",\
      "AdultHeartSmoothMuscle":"AH smooth musc.", "FetalBrainRoofPlate":"FB ro. plate"}


In [ ]:
blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
            "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
            "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

#Genes that are near or in IGV etc. areas
blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]

#Genes that seem a little iffy since they are embedded in regions with lots of the same gene
iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
orth = orth[~orth["Gene name"].isin(blacklist + blacklist2)]

olfs = []
hlas = []
ugt = []
for i in orth["Gene name"]:
    try:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    except:
        pass
orth = orth[~orth["Gene name"].isin(olfs + hlas + ugt)]
pd.DataFrame(orth.drop_duplicates("Gene name")["Gene name"]).to_csv("Whitelist_Genes.txt", sep = "\t", header = False, index = False)
bad_ml = list(pd.read_csv("Blacklist_Chimp_NonCanonicalChrom.txt", sep = "\t", header = None)[0])
pd.DataFrame(orth[~orth["Gene name"].isin(bad_ml)].drop_duplicates("Gene name")["Gene name"]).to_csv("Whitelist_Genes_ForML.txt", sep = "\t", header = False, index = False)
pd.DataFrame(orth.drop_duplicates("Gene name")["Gene name"])

In [ ]:
def filter_nc(file):
    v = pd.read_csv(file, sep = "\t")
    try:
        v = v[['NearestGene', 'Species1 Sum abs logfc', 'Species1 Sum Total_Vars', 'Species2 Sum abs logfc', 'Species2 Sum Total_Vars', 'Abs Logfc Difference', 'Z-score Difference', 'p-value Difference', 'Z-score Difference Corr Tot', 'p-value Difference Corr Tot', 'Z-score Difference Corr S', 'p-value Difference Corr S', 'FDR Difference', 'FDR Difference Corr Tot', 'FDR Difference Corr S']]
    except:
        v = v[['NearestGene', 'Species1 Sum logfc', 'Species1 Sum Total_Vars', 'Species2 Sum logfc', 'Species2 Sum Total_Vars', 'Logfc Difference', 'Z-score Difference', 'p-value Difference', 'Z-score Difference Corr Tot', 'p-value Difference Corr Tot', 'Z-score Difference Corr S', 'p-value Difference Corr S', 'FDR Difference', 'FDR Difference Corr Tot', 'FDR Difference Corr S']]

    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP", "SULT1C3"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["NearestGene"].isin(orth["Gene name"])]
    v = v[~v["NearestGene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["NearestGene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)

    #Remove olfactory receptors and HLA genes
    v = v[~v["NearestGene"].isin(olfs + hlas + ugt)]

    #Remove the ML-specific blacklisted genes
    bad_ml = list(pd.read_csv("Blacklist_Chimp_NonCanonicalChrom.txt", sep = "\t", header = None)[0])
    v = v[~v["NearestGene"].isin(bad_ml)]
    
    #Filter to only genes with sufficient nearby substituions
    v = v[(v["Species1 Sum Total_Vars"] > 10) | (v["Species2 Sum Total_Vars"] > 10)]

    #Recompute FDR after filtering
    v["FDR Difference"] = fdrcorrection(v["p-value Difference"])[1]
    v["FDR Difference Corr Tot"] = fdrcorrection(v["p-value Difference Corr Tot"])[1]
    v["FDR Difference Corr S"] = fdrcorrection(v["p-value Difference Corr S"])[1]
    
    return v.sort_values("p-value Difference Corr Tot")

In [ ]:
def mvp(v, xlab, title, stat = 'Abs Logfc Difference', fdr_col = "FDR Difference", x_col = 'Abs Logfc Difference'):
    plt.figure(figsize=(6, 4), dpi=600)
    sig = []
    #Also filtered out sites that were variable in apes or variable across human populations
    for index, row in v.iterrows():
        if row[fdr_col] < 0.05 and row[stat] < 0:
            sig.append("Chimp accelerated")
        elif row[fdr_col] < 0.05 and row[stat] > 0:
            sig.append("Human accelerated")
        else:
            sig.append("Not significant")
    v["Category"] = sig
    v["-Log$_{10}$(FDR)"] = -np.log10(v[fdr_col])
    v = v.sort_values("Category")
    sns.scatterplot(data = v, x = stat, y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})

    xlim1 = min(v[stat]) - abs(floor(0.1*min(v[stat])))
    xlim2 = max(v[stat]) + abs(floor(0.1*max(v[stat])))
    plt.xlim(xlim1, xlim2)
    plt.hlines(-np.log10(0.05), xlim1, xlim2, linestyles="dashed", colors="k")
    plt.legend(bbox_to_anchor = (1, 1))
    plt.xlabel(xlab, size = 13)
    plt.ylabel("-Log$_{10}$(FDR)", size = 13)
    plt.xticks(size = 12)
    plt.yticks(size = 12)
    plt.title(title, size = 14)
    plt.legend([], [], frameon = False)
    plt.show()

In [ ]:
v = filter_nc("ML_Results_Download/FetalBrainNeurPurkinje_6_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt")
v2 = filter_nc("ML_Results_Download/FetalBrainNeurPurkinje_6_AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt")

sig = np.intersect1d(v[(v["FDR Difference"] < 0.05) & (v["FDR Difference Corr Tot"] < 0.05)]["NearestGene"], v2[(v2["FDR Difference"] < 0.25) & (v2["FDR Difference Corr Tot"] < 0.25)]["NearestGene"])
nsig = np.intersect1d(v[(v["FDR Difference"] > 0.05) & (v["FDR Difference Corr Tot"] > 0.05)]["NearestGene"], v2[(v2["FDR Difference"] > 0.05) & (v2["FDR Difference Corr Tot"] > 0.05)]["NearestGene"])
v_sig = v[v["NearestGene"].isin(sig)]
v_nsig = v[v["NearestGene"].isin(nsig)]
v2_sig = v2[v2["NearestGene"].isin(sig)]
v2_nsig = v2[v2["NearestGene"].isin(nsig)]

v_filt = pd.concat([v_sig, v_nsig])
v2_filt = pd.concat([v2_sig, v2_nsig])
mvp(v_filt, xlab = "Bleh", title = "Bleh")

In [ ]:
v = filter_nc("ML_PhyloP1/FetalBrainNeurPurkinje_6_AbsLogfc_NC_FiltWGS_PhyloP1_SpecSup250_YCM_NCV_Per90_PerGene_Results.txt")
v[(v["p-value Difference"] < 0.05) & (v["Z-score Difference"] > 0)].shape

In [ ]:
out = []
for file in os.listdir("ML_PhyloP1"):
    if "NoFiltWGS" in file:
        
        if file.split("_Abs")[0] not in ["FetalHeartCardiacFibroblasts", "FetalHeartPericytes", "FetalHeartEndocardium", 'AdultHeartSmoothMuscle']:
            v = filter_nc("ML_PhyloP1/" + file)
            v2 = filter_nc("ML_PhyloP1/" + file.replace("NoFiltWGS", "FiltWGS"))
            ha = v[(v["Z-score Difference Corr Tot"] > 2) & (v["Z-score Difference"] > 2)].shape[0]
            ca = v[(v["Z-score Difference Corr Tot"] < -2) & (v["Z-score Difference"] < -2)].shape[0]
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "NoFiltWGS", "PhyloP1"])
            ha = v2[(v2["Z-score Difference Corr Tot"] > 2) & (v2["Z-score Difference"] > 2)].shape[0]
            ca = v2[(v2["Z-score Difference Corr Tot"] < -2) & (v2["Z-score Difference"] < -2)].shape[0]
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "FiltWGS", "PhyloP1"])
    
            v = filter_nc("ML_Results_Download/" + file.replace("P1", "P-100").replace("Sup250", "Sup0"))
            v2 = filter_nc("ML_Results_Download/" + file.replace("NoFiltWGS", "FiltWGS").replace("P1", "P-100").replace("Sup250", "Sup0"))
            ha = v[(v["Z-score Difference Corr Tot"] > 2) & (v["Z-score Difference"] > 2)].shape[0]
            ca = v[(v["Z-score Difference Corr Tot"] < -2) & (v["Z-score Difference"] < -2)].shape[0]
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "NoFiltWGS", "PhyloP-100"])
            ha = v2[(v2["Z-score Difference Corr Tot"] > 2) & (v2["Z-score Difference"] > 2)].shape[0]
            ca = v2[(v2["Z-score Difference Corr Tot"] < -2) & (v2["Z-score Difference"] < -2)].shape[0]
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "FiltWGS", "PhyloP-100"])

df = pd.DataFrame(out)
df.columns = ["Cell type", "Human accelerated", "Chimp accelerated", "p-value", "Filtering", "PhyloP"]
df1 = df[(df["Filtering"] == "NoFiltWGS") & (df["PhyloP"] == "PhyloP1")]
df2 = df[(df["Filtering"] == "NoFiltWGS") & (df["PhyloP"] == "PhyloP-100")]
df3 = df[(df["Filtering"] == "FiltWGS") & (df["PhyloP"] == "PhyloP1")]
df4 = df[(df["Filtering"] == "FiltWGS") & (df["PhyloP"] == "PhyloP-100")]
df1["FDR"] = fdrcorrection(df1["p-value"])[1]
df2["FDR"] = fdrcorrection(df2["p-value"])[1]
df3["FDR"] = fdrcorrection(df3["p-value"])[1]
df4["FDR"] = fdrcorrection(df4["p-value"])[1]
df = pd.concat([df1, df2, df3, df4])
dff = df[df["Filtering"] == "NoFiltWGS"]
dff.sort_values("FDR")

In [ ]:
out = []
hac = []
cac = []
hanc = []
canc = []
for file in os.listdir("ML_PhyloP1"):
    if "NoFiltWGS" in file:
        
        if file.split("_Abs")[0] not in ["FetalHeartCardiacFibroblasts", "FetalHeartPericytes", "FetalHeartEndocardium", 'AdultHeartSmoothMuscle']:
            v = filter_nc("ML_PhyloP1/" + file)
            v2 = filter_nc("ML_PhyloP1/" + file.replace("NoFiltWGS", "FiltWGS"))
            ha = len(np.intersect1d(v[(v["Z-score Difference Corr Tot"] > 1.96) & (v["Z-score Difference"] > 1.96)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] > 1.15) & (v2["Z-score Difference"] > 1.15)]["NearestGene"]))
            ca = len(np.intersect1d(v[(v["Z-score Difference Corr Tot"] < -1.96) & (v["Z-score Difference"] < -1.96)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] < -1.15) & (v2["Z-score Difference"] < -1.15)]["NearestGene"]))
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "Combined", "PhyloP1"])
    
            v = filter_nc("ML_Results_Download/" + file.replace("P1", "P-100").replace("Sup250", "Sup0"))
            v2 = filter_nc("ML_Results_Download/" + file.replace("NoFiltWGS", "FiltWGS").replace("P1", "P-100").replace("Sup250", "Sup0"))
            ha = len(np.intersect1d(v[(v["Z-score Difference Corr Tot"] > 2) & (v["Z-score Difference"] > 2)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] > 1.15) & (v2["Z-score Difference"] > 1.15)]["NearestGene"]))
            ca = len(np.intersect1d(v[(v["Z-score Difference Corr Tot"] < -2) & (v["Z-score Difference"] < -2)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] < -1.15) & (v2["Z-score Difference"] < -1.15)]["NearestGene"]))

            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, binomtest(ha, ha + ca).pvalue, "Combined", "PhyloP-100"])

df = pd.DataFrame(out)
df.columns = ["Cell type", "Human accelerated", "Chimp accelerated", "p-value", "Filtering", "PhyloP"]
df1 = df[(df["Filtering"] == "Combined") & (df["PhyloP"] == "PhyloP1")]
df2 = df[(df["Filtering"] == "Combined") & (df["PhyloP"] == "PhyloP-100")]
df1["FDR"] = fdrcorrection(df1["p-value"])[1]
df2["FDR"] = fdrcorrection(df2["p-value"])[1]
df = pd.concat([df1, df2])
dff = df[df["Filtering"] == "Combined"]
dff.sort_values("FDR")

In [ ]:
dff["log2(human/chimp)"] = np.log2(df["Human accelerated"]/df["Chimp accelerated"])
dff = dff.sort_values("FDR")
x = []
for index, row in dff.iterrows():
    if row["FDR"] < 0.05 and row["PhyloP"] == "PhyloP1":
        x.append("Significant1")
    elif row["PhyloP"] == "PhyloP1":
        x.append("Not significant1")
    elif row["FDR"] < 0.05 and row["PhyloP"] == "PhyloP-100":
        x.append("Significant2")
    elif row["PhyloP"] == "PhyloP-100":
        x.append("Not significant2")
dff["Sig"] = x
plt.figure(figsize=(12, 4), dpi=600)
color_map = {
    "Significant1": palette["Human"],
    "Significant2": palette["Chimp"],
    "Not significant1": "#808080",
    "Not significant2": "#A9A9A9"
}

dff["color"] = dff["Sig"].map(color_map)
ax = sns.barplot(data = dff, x = "Cell type", y = "log2(human/chimp)", width = 0.9, hue = "PhyloP")
for bar, color in zip(ax.patches, dff["color"]):
    bar.set_facecolor(color)
plt.ylabel("Log$_{2}$(human/chimp)", size = 14)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("More CA acceleration in human only in conserved sites", size=16)
plt.legend([], [], frameon = False)
plt.xticks(rotation = 90)

In [ ]:
plt.figure(figsize=(3, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [77, 31], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of genes with accelerated\nCA evolution in Purkinje cells", size = 12)
plt.ylim(0, 95)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Conserved sites", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 85, 0.6             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(3, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [205, 209], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of genes with nominally accelerated\nCA evolution in Purkinje cells", size = 12)
plt.ylim(0, 250)
plt.xticks(size = 14)
plt.yticks(size = 12)
plt.title("All sites", size = 15)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 220, 1.8             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "ns",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
z = pd.read_csv("FetalBrainNeurPurkinje_6_AllSitesToDownload.txt.gz", sep = "\t")
z

In [ ]:
z["AncNuc"] = [x[1] for x in z["AncTrinuc"]]
z["DerNuc"] = [x[1] for x in z["DerTrinuc"]]


In [ ]:
z2 = z[z["SpecSup447"] > 250]
z2 = z[z["PhyloP447"] > 3]
z2 = z2[z2["AncNuc"] == "A"]
sns.histplot(z2["PhyloP447"])

In [ ]:
z2 = z[z["NearestDist"] < 5000]
z2 = z2[(z2["SpecSup447"] > 250) & (z2["PhyloP447"] > 4)]
z2h = z2[z2["Derived"] == "H"]
z2c = z2[z2["Derived"] == "C"]
hd = z2h[z2h["logfc"] > 0.25].shape[0]
hu = z2h[z2h["logfc"] < -0.25].shape[0]
cd = z2c[z2c["logfc"] > 0.25].shape[0]
cu = z2c[z2c["logfc"] < -0.25].shape[0]
OR = fisher_exact([[hd, hu], [cd, cu]])[0]
p = fisher_exact([[hd, hu], [cd, cu]])[1]
print([hd, hu, cd, cu, OR, p, "Signed"])

In [ ]:
out = []
for i in [0, 1, 2, 3, 4, 5, 6]:
    if i != 6:
        z2 = z[(z["SpecSup447"] > 250) & (z["PhyloP447"] < i + 1) & (z["PhyloP447"] > i) & ((z["AncNuc"] == "A") | (z["AncNuc"] == "T"))]
    else:
        z2 = z[(z["SpecSup447"] > 250) & (z["PhyloP447"] > i + 1) & ((z["AncNuc"] == "A") | (z["AncNuc"] == "T"))]
    z2h = z2[(z2["Derived"] == "H") & (z2["KeptAfterFilt"] == "Y")]
    z2c = z2[(z2["Derived"] == "C") & (z2["KeptAfterFilt"] == "Y")]
    z2h = z2[(z2["Derived"] == "H")]
    z2c = z2[(z2["Derived"] == "C")]
    out.append([z2h.shape[0], z2c.shape[0]])



In [ ]:
#Checking that this is not the result of ChromBPNet generally predictin weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result generally gets stronger at 0.5 rather than 0.25

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("GC -> AT", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
j = pd.read_csv("../PosSelect_ForPub/AllNonPoly_Fixed.NPC_Cis_JanetSong.bed", sep = "\t", header = None)


In [ ]:
j = j[j[5] != -1]
j.index = j[0] + ":" + j[2].astype(str)
j = j[[26, 27]].join(z.set_index("Position.1"))

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result generally gets stronger at 0.5 rather than 0.25
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("GC -> GC", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result becomes significant, but does get stronger for this too
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("AT -> GC", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Stronger at 0.5, but so few subs that it is hard to evaluate
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.25].shape[0]
down_cons = z2[z2["logfc"] > 0.25].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("AT -> AT", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
sns.barplot(x = ["Human-derived", "Chimp-derived"], y = [17465, 14383], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of subs. in highly conserved sites", size = 12)
plt.ylim(0, 20000)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Human-derived subs. are\ndisproportionately in conserved sites", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 18000, 400             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h + 150,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
df = pd.DataFrame(out)
df.columns = ["Human", "Chimp"]
df["Log$_{2}$(human/chimp)"] = np.log2(df["Human"]/df["Chimp"])
o = []
for index, row in df.iterrows():
    o.append(binomtest(int(row["Human"]), int(row["Chimp"]) + int(row["Human"])).pvalue)
df["p-value"] = o
df["PhyloP bin"] = ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6", "> 6"]
df["Hue"] = ["Chimp", "Not significant", "Human", "Human", "Human", "Human", "Human"]
sns.barplot(df, y = "Log$_{2}$(human/chimp)", x = "PhyloP bin", hue = "Hue", palette = palette, alpha = 0.8)
plt.xlabel("PhyloP bin", size = 13)
plt.ylabel("Log$_{2}$(human/chimp)", size = 13)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Human-derived subs. are\ndisproportionately in conserved sites", size = 12)
plt.legend([], [], frameon = False)
plt.show()

In [ ]:
v = pd.read_csv("Phylo_Stuff.txt", sep = "\t")
keep = []
kk = []
for i in np.unique(v["CellType"]):
    vk = v[v["CellType"] == i]
    vk = vk[vk["LFC Left Bin"] == 0.5]
    kk.append([i, np.sum(np.sum(vk[["Human down", "Human up", "Chimp down", "Chimp up"]]))])
    if np.sum(np.sum(vk[["Human down", "Human up", "Chimp down", "Chimp up"]])) >= 10000:
        keep.append(i)
        
v = v[v["CellType"].isin(keep)]
v

In [ ]:

vk = v[v["CellType"] == "FetalBrainNeurPurkinje_6"]
vk = vk[vk["FiltOrNotFilt"] == "NotFilt"]
vk2 = vk[vk["LFC Left Bin"] == 0.5]
vk3 = vk[vk["LFC Left Bin"] == 0.25]
cols = ["Human down", "Human up", "Chimp down", "Chimp up"]

grouped = (
    vk3
    .groupby("PhyloP Left Bin", as_index=False)[cols]
    .sum()
)

In [ ]:
grouped["Down"] = np.sum(grouped[["Human down", "Chimp down"]], axis = 1)
grouped["Up"] = np.sum(grouped[["Human up", "Chimp up"]], axis = 1)
df = grouped.copy()

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
df["Log$_{2}$(decreasing CA/increasing CA)"] = np.log2(df["Down"]/df["Up"])
o = []
for index, row in df.iterrows():
    o.append(binomtest(int(row["Down"]), int(row["Up"]) + int(row["Down"])).pvalue)
df["p-value"] = o
df["PhyloP bin"] = ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6", "> 6"]
#For 0.5
df["Hue"] = ["Sig", "Sig", "Sig", "Sig", "Sig", "Sig", "Sig"]
#For 0.25
df["Hue"] = ["Up", "Not significant", "Sig", "Sig", "Sig", "Sig", "Sig"]

#df["Hue"] = ["Up", "Not significant", "Sig", "Sig", "Sig", "Sig", "Sig"]
sns.barplot(df, y = "Log$_{2}$(decreasing CA/increasing CA)", x = "PhyloP bin", hue = "Hue", palette = {"Not significant":"grey", "Sig":"magenta", "Up":"green"}, alpha = 0.8)
plt.xlabel("PhyloP bin", size = 11)
plt.ylabel("Log$_{2}$(decreasing CA/increasing CA)", size = 11)
plt.xticks(size = 11)
plt.yticks(size = 11)
plt.title("Bias toward decreasing CA in conserved sites\n0.25 < |log$_{2}$ fold-change| < 0.5", size = 12)
plt.legend([], [], frameon = False)
plt.show()

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
df["Log$_{2}$(decreasing CA/increasing CA)"] = np.log2(df["Down"]/df["Up"])
o = []
for index, row in df.iterrows():
    o.append(binomtest(int(row["Down"]), int(row["Up"]) + int(row["Down"])).pvalue)
df["p-value"] = o
df["PhyloP bin"] = ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6", "> 6"]
#For 0.5
df["Hue"] = ["Sig", "Sig", "Sig", "Sig", "Sig", "Sig", "Sig"]
#For 0.25
#df["Hue"] = ["Up", "Not significant", "Sig", "Sig", "Sig", "Sig", "Sig"]
sns.barplot(df, y = "Log$_{2}$(decreasing CA/increasing CA)", x = "PhyloP bin", hue = "Hue", palette = {"Not significant":"grey", "Sig":"magenta", "Up":"green"}, alpha = 0.8)
plt.xlabel("PhyloP bin", size = 11)
plt.ylabel("Log$_{2}$(decreasing CA/increasing CA)", size = 11)
plt.xticks(size = 11)
plt.yticks(size = 11)
plt.title("Bias toward decreasing CA in conserved sites", size = 12)
plt.legend([], [], frameon = False)
plt.show()

In [ ]:
plt.figure(figsize=(3, 4), dpi=600)

sns.barplot(x = ["Decrease CA", "Increase CA"], y = [492, 168], hue = ["Decrease CA", "Increase CA"], palette = {"Decrease CA":"magenta", "Increase CA":"green"}, alpha = 0.8)
plt.ylabel("Number of subs. affecting CA", size = 11)
plt.ylim(0, 555)
plt.xticks(size = 11)
plt.yticks(size = 11)
plt.title("Subs. in highly conserved sites\ntend to decrease CA", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 500, 0             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h - 5,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
binomtest(492, 492 + 168)

In [ ]:
plt.figure(figsize=(3, 4), dpi=600)

sns.barplot(x = ["Decrease CA", "Increase CA"], y = [1046, 690], hue = ["Decrease CA", "Increase CA"], palette = {"Decrease CA":"magenta", "Increase CA":"green"}, alpha = 0.8)
plt.ylim(0, 1200)
plt.ylabel("Number of subs. affecting CA", size = 12)
plt.xlabel("0.25 < |log$_{2}$ fold-change| < 0.5", size = 12)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Subs in highly conserved sites\ntend to decrease CA", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 1075, 0             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
#Defining neural, glial/progenitor, and non-nervous system cell types
neur = ['FC exc. neur.', 'FG neur.', 'AC VIP inh. neur.', 'AC SST inh neur.', 'AA neur.', 'AC PVALB inh neur.', 'AC L2-3 IT neur.', 'FB DA neur.', 'AP D1 inh neur.', \
       'FB 5-HT neur.', 'FS DRG neur.', 'FCB Purk. inh neur.']
glia = ['AC microglia', 'FC prog.', 'FCB glioblast', 'AC astro.', 'FB COP', 'FB inter. prog.', 'AC Oligo.', 'FB ro. plate', 'FB fl. plate']
nneur = np.setdiff1d(list(d_abrev.values()), neur + glia)

In [ ]:
#Pulling out the nominally accelerated ones
out = []
hac = []
cac = []
hanc = []
canc = []
for file in os.listdir("ML_PhyloP1"):
    if "NoFiltWGS" in file:
        
        if file.split("_Abs")[0] not in ["FetalHeartCardiacFibroblasts", "FetalHeartPericytes", "FetalHeartEndocardium", 'AdultHeartSmoothMuscle']:
            v = filter_nc("ML_PhyloP1/" + file)
            v2 = filter_nc("ML_PhyloP1/" + file.replace("NoFiltWGS", "FiltWGS"))
            ha = ";".join(np.intersect1d(v[(v["Z-score Difference Corr Tot"] > 1.96) & (v["Z-score Difference"] > 1.96)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] > 1.15) & (v2["Z-score Difference"] > 1.15)]["NearestGene"]))
            ca = ";".join(np.intersect1d(v[(v["Z-score Difference Corr Tot"] < -1.96) & (v["Z-score Difference"] < -1.96)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] < -1.15) & (v2["Z-score Difference"] < -1.15)]["NearestGene"]))
            na = ";".join(np.setdiff1d(np.intersect1d(v["NearestGene"], v2["NearestGene"]), ha.split(";") + ca.split(";")))
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, na, "Combined", "PhyloP1"])
    
            v = filter_nc("ML_Results_Download/" + file.replace("P1", "P-100").replace("Sup250", "Sup0"))
            v2 = filter_nc("ML_Results_Download/" + file.replace("NoFiltWGS", "FiltWGS").replace("P1", "P-100").replace("Sup250", "Sup0"))
            ha = ";".join(np.intersect1d(v[(v["Z-score Difference Corr Tot"] > 2) & (v["Z-score Difference"] > 2)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] > 1.15) & (v2["Z-score Difference"] > 1.15)]["NearestGene"]))
            ca = ";".join(np.intersect1d(v[(v["Z-score Difference Corr Tot"] < -2) & (v["Z-score Difference"] < -2)]["NearestGene"], v2[(v2["Z-score Difference Corr Tot"] < -1.15) & (v2["Z-score Difference"] < -1.15)]["NearestGene"]))
            na = ";".join(np.setdiff1d(np.intersect1d(v["NearestGene"], v2["NearestGene"]), ha.split(";") + ca.split(";")))
            out.append([d_abrev[file.split("_Abs")[0]], ha, ca, na, "Combined", "PhyloP-100"])

df = pd.DataFrame(out)
df.columns = ["Cell type", "Human accelerated", "Chimp accelerated", "Not accelerated", "Filtering", "PhyloP"]

In [ ]:
b = pd.read_csv("SFARI-Gene_genes_03-28-2024release_05-09-2024export.csv")
hall = []
call = []
nall = []
for index, row in df.iterrows():
    if row["PhyloP"] == "PhyloP1" and row["Cell type"] in nneur:
        h = row["Human accelerated"].split(";")
        hall = h + hall
        c = row["Chimp accelerated"].split(";")
        call = c + call
        n = row["Not accelerated"].split(";")
        nall = n + nall

b = pd.read_csv("SFARI-Gene_genes_03-28-2024release_05-09-2024export.csv")
nhall = []
ncall = []
nnall = []
for index, row in df.iterrows():
    if row["PhyloP"] == "PhyloP-100" and row["Cell type"] in nneur:
        h = row["Human accelerated"].split(";")
        nhall = h + nhall
        c = row["Chimp accelerated"].split(";")
        ncall = c + ncall
        n = row["Not accelerated"].split(";")
        nnall = n + nnall

In [ ]:
htemp = hall
hall = np.unique(np.setdiff1d(hall, call))
call = np.unique(np.setdiff1d(call, htemp))
nall = np.unique(np.setdiff1d(np.setdiff1d(nall, hall), call))

nhtemp = nhall
nhall = np.unique(np.setdiff1d(nhall, ncall))
ncall = np.unique(np.setdiff1d(ncall, nhtemp))
nnall = np.unique(np.setdiff1d(np.setdiff1d(nnall, nhall), ncall))

print(len(hall), len(call), len(nall))
print(len(nhall), len(ncall), len(nnall))

In [ ]:
ha = len(np.intersect1d(b1["gene-symbol"], np.setdiff1d(hall, nhall)))
ca = len(np.intersect1d(b1["gene-symbol"], np.setdiff1d(call, ncall)))
hna = len(np.setdiff1d(np.setdiff1d(hall, nhall), b1["gene-symbol"]))
cna = len(np.setdiff1d(np.setdiff1d(call, ncall), b1["gene-symbol"]))

print([[ha, ca], [hna, cna]])


In [ ]:
#Conserved neuronal
print(binomtest(34, 16 + 34))
#Unconserved neuronal
print(binomtest(82, 82 + 85))
#Conserved glia/prog
print(binomtest(23, 23 + 30))
#Unconserved glia/prog
print(binomtest(73, 73 + 67))
#Conserved nneur
print(binomtest(22, 22 + 12))
#Unconserved nneur
print(binomtest(96, 96 + 91))

In [ ]:
plt.figure(figsize=(1.5, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [22, 12], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of ASD-linked genes with nominally\naccelerated CA evolution in neurons", size = 12)
plt.ylim(0, 27)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Conserved sites\nnon-nervous", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 24, 0            # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y,
         "NS",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
#Defining synaptic stuff
ca_pre_syn = ["AGAP1", "CACNA2D1", "CACNA1A", "GABBR2", "GRIN2B", "GRM7", "LRRC4C", "NRXN3", "PCDH9"]
ca_post_syn = ["CACNA2D1", "CACNA1A", "CNKSR2" "DLG1", "GABBR2", "GRIN2B", "KCNB1", "LRRC4C", "NEGR1", "NRXN3", "PCDH9", "PRICKLE2", "SDC2"]
broad_syn = ['ANK2', 'CNTN3', 'CNTNAP5', 'SNCAIP', 'ST8SIA2', 'UNC80']
ca_nuc = ['AR', 'CELF4', 'CHD1', 'MED13L', 'MSX2', 'PTBP2', 'SATB2', "SETBP1", 'TNRC6C', 'ZMYND8']
ca_other = ['ATP1A1', 'PEBP4', 'REEP3']

p_pre_syn = ["CDH11", "CDH13", "GRIK3", "NTNG1"]
p_post_syn = ["CDH11", "CDH13", "FGFR1", "GRIK3", "MARK1", "NEO1", "NTNG1"]
p_nuc = ['BCL11B', 'CASZ1', 'DLX6', 'EBF3', 'FEZF2', 'FOXP1', 'RBFOX1', 'RORB', 'RUNX1T1', 'SMG6', 'TCF7L2', 'TLE3', 'TSHZ3', 'ZBTB16']
p_other = ['TBC1D5', 'WWOX']

In [ ]:
x = pd.read_csv("TestingConsEnrichment.csv")
x["Cell type"] = [d_abrev[i] for i in x["Cell type"]]
x.columns = list(x.columns)[0:5] + ["Odds ratio", "p-value", "Test", "Bin"]
x2 = x[(x["Bin"].isin(["(9, 12)", "(6, 9)"])) & (x["Test"] == "Total CA")].sort_values("Odds ratio")
x2_new = (
    x2
    .groupby("Cell type", as_index=False)
    .agg({
        "Conserved_Tail": "sum",
        "Conserved_Head": "sum",
        **{col: "first" for col in x2.columns if col not in ["bin", "Conserved_Tail", "Conserved_Head"]}
    })
)

out = []
for index, row in x2_new.iterrows():
    fe = fisher_exact([[row["Conserved_Tail"], row["Conserved_Head"]], [row["Unconserved_Tail"], row["Unconserved_Head"]]])
    out.append([row["Cell type"], fe[0], fe[1]])
dfp = pd.DataFrame(out).sort_values(1)
dfp.columns = ["Cell type", "Odds ratio", "p-value"]
dfp["FDR"] = fdrcorrection(dfp["p-value"])[1]


In [ ]:
lab = []
for index, row in dfp.iterrows():
    if row["Cell type"] in neur:
        lab.append("Neuron")
    elif row["Cell type"] in glia:
        lab.append("Glia or progenitor")
    elif row["Cell type"] in nneur:
        lab.append("Non-neuronal")
dfp["Category"] = lab